# 08 - Publication Grade Experiments

Run this after `00_MASTER_RUN_ALL.ipynb` and the review notebooks have produced clean Coswara artifacts. This notebook adds the publication-facing reliability experiments: COUGHVID indexing, metadata-only baseline, quality-weighted fusion, abstention, bootstrap confidence intervals, and optional cross-dataset cough evaluation.

Keep this notebook as an experiment coordinator. Core logic stays in `src/covid_audio_btp/` and scripts stay in `scripts/`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    check_artifacts,
    read_optional_csv,
    save_table,
)

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
METRICS_DIR = DATA_DIR / "outputs" / "metrics"
INTERIM_DIR = DATA_DIR / "interim"

print(f"Project root: {PROJECT_ROOT}")

## Configuration

Set `COUGHVID_RAW` to either an extracted COUGHVID directory or the official `public_dataset.zip`. Leaving it as `None` skips COUGHVID indexing.

In [ ]:
COUGHVID_RAW = None  # Example: Path("/tmp/coughvid/public_dataset.zip") or Path("/data/coughvid")
MIN_COUGH_DETECTED = 0.80
N_BOOTSTRAPS = 1000
RANDOM_STATE = 42

RUN_METADATA_BASELINE = True
RUN_QUALITY_WEIGHTED_FUSION = True
RUN_ABSTENTION = True
RUN_BOOTSTRAP_CI = True
RUN_CROSS_DATASET = False

SOURCE_FEATURES = DATA_DIR / "processed" / "features_mfcc.csv"
EXTERNAL_FEATURES = DATA_DIR / "processed" / "coughvid_features_mfcc.csv"

## Artifact Gate

The publication experiments should not hide missing upstream artifacts. This table shows what exists before running optional cells.

In [ ]:
status = check_artifacts([
    "data/processed/metadata_clean.csv",
    "data/processed/audio_quality.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/coughvid_features_mfcc.csv",
])
display(status)

## COUGHVID Index

The adapter supports the official sidecar layout `public_dataset/<uuid>.json` plus `.webm`/`.ogg` audio, direct zip inspection, and CSV metadata such as `metadata_compiled.csv`.

In [ ]:
from covid_audio_btp.external_datasets import build_coughvid_index

if COUGHVID_RAW is None:
    print("COUGHVID_RAW is not set; skipping external index build.")
else:
    raw_path = Path(COUGHVID_RAW)
    if not raw_path.exists():
        raise FileNotFoundError(raw_path)
    coughvid_index = build_coughvid_index(raw_path, min_cough_detected=MIN_COUGH_DETECTED)
    save_table(coughvid_index, "data/interim/coughvid_index.csv")
    display(coughvid_index.head())
    display(coughvid_index["label_binary"].value_counts(dropna=False).rename_axis("label_binary").reset_index(name="n"))
    display(coughvid_index["manual_quality_label"].value_counts(dropna=False).rename_axis("manual_quality_label").reset_index(name="n"))

## COUGHVID Feature Extraction

Use `MAX_EXTERNAL_ROWS` for a quick smoke test before running the full COUGHVID feature extraction. Direct zip paths are supported, but extracting the archive once is faster for full runs.

In [ ]:
from covid_audio_btp.external_features import write_external_feature_table

MAX_EXTERNAL_ROWS = None  # Example: 25 for a smoke test before the full run
RUN_COUGHVID_FEATURES = False

coughvid_index_path = INTERIM_DIR / "coughvid_index.csv"
if RUN_COUGHVID_FEATURES:
    if not coughvid_index_path.exists():
        raise FileNotFoundError("Build data/interim/coughvid_index.csv before extracting COUGHVID features.")
    coughvid_features = write_external_feature_table(
        coughvid_index_path,
        DATA_DIR / "processed" / "coughvid_features_mfcc.csv",
        split_name="external",
        labeled_only=True,
        quality_ok_only=True,
        max_rows=MAX_EXTERNAL_ROWS,
    )
    display(coughvid_features.head())
else:
    print("RUN_COUGHVID_FEATURES is False; skipping external feature extraction.")

## Metadata Baseline

This is a confounding guardrail. If symptom/demographic metadata alone performs strongly, the audio model claims must be framed cautiously.

In [ ]:
from covid_audio_btp.metadata_baseline import train_metadata_baseline

metadata = read_optional_csv("data/processed/metadata_clean.csv", n=3)
if RUN_METADATA_BASELINE and metadata is not None:
    result = train_metadata_baseline(metadata)
    metrics = pd.DataFrame([result.metrics])
    save_table(metrics, "data/outputs/metrics/metadata_baseline_metrics.csv")
    save_table(result.validation_predictions, "data/outputs/metrics/metadata_baseline_validation_predictions.csv")
    save_table(result.test_predictions, "data/outputs/metrics/metadata_baseline_test_predictions.csv")
    display(metrics)
else:
    print("Skipping metadata baseline.")

## Quality-Weighted Fusion

This replaces naive equal averaging with validation-performance and quality-aware branch weighting. It must be compared against the best single branch.

In [ ]:
from covid_audio_btp.fusion import quality_weighted_fusion

predictions = read_optional_csv("data/outputs/metrics/calibrated_branch_predictions.csv", n=3)
quality = read_optional_csv("data/processed/audio_quality.csv", n=3)
validation_metrics = read_optional_csv("data/outputs/metrics/ml_baseline_metrics.csv", n=3)

if RUN_QUALITY_WEIGHTED_FUSION and predictions is not None and quality is not None and validation_metrics is not None:
    fused = quality_weighted_fusion(predictions, quality, validation_metrics)
    save_table(fused, "data/outputs/metrics/quality_weighted_fusion_predictions.csv")
    display(fused.head())
else:
    print("Skipping quality-weighted fusion until calibrated predictions, quality table, and validation metrics exist.")

## Abstention and Coverage

This quantifies the tradeoff between coverage and reliability when uncertain or low-quality samples are rejected.

In [ ]:
from covid_audio_btp.abstention import apply_abstention, coverage_curve

fusion_predictions = read_optional_csv("data/outputs/metrics/quality_weighted_fusion_predictions.csv", n=3)
if RUN_ABSTENTION and fusion_predictions is not None:
    abstention_input = fusion_predictions.copy()
    if "quality_flag" not in abstention_input.columns:
        quality = read_optional_csv("data/processed/audio_quality.csv")
        if quality is not None and "recording_id" in abstention_input.columns:
            abstention_input = abstention_input.merge(quality[["recording_id", "quality_flag"]], on="recording_id", how="left")
    abstained = apply_abstention(abstention_input)
    curve = coverage_curve(abstained)
    save_table(abstained, "data/outputs/metrics/abstention_predictions.csv")
    save_table(curve, "data/outputs/metrics/abstention_coverage_curve.csv")
    display(curve)
else:
    print("Skipping abstention analysis.")

## Bootstrap Confidence Intervals

Use confidence intervals in the paper tables instead of reporting point estimates only.

In [ ]:
from covid_audio_btp.statistics import bootstrap_prediction_table

ci_input = read_optional_csv("data/outputs/metrics/quality_weighted_fusion_predictions.csv", n=3)
if RUN_BOOTSTRAP_CI and ci_input is not None:
    group_columns = ["fusion_method"] if "fusion_method" in ci_input.columns else []
    ci = bootstrap_prediction_table(
        ci_input,
        group_columns=group_columns,
        n_bootstraps=N_BOOTSTRAPS,
        random_state=RANDOM_STATE,
    )
    save_table(ci, "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv")
    display(ci)
else:
    print("Skipping bootstrap confidence intervals.")

## Optional Cross-Dataset Cough Evaluation

Run only after both Coswara cough features and COUGHVID cough features exist. This is intentionally a hard external validation, not mixed-dataset training.

In [ ]:
from covid_audio_btp.cross_dataset import harmonize_feature_columns
from covid_audio_btp.metrics import evaluate_predictions, labels_to_binary
from covid_audio_btp.models_ml import make_model

if RUN_CROSS_DATASET:
    if not SOURCE_FEATURES.exists() or not EXTERNAL_FEATURES.exists():
        raise FileNotFoundError("Both SOURCE_FEATURES and EXTERNAL_FEATURES are required for cross-dataset evaluation.")
    source = pd.read_csv(SOURCE_FEATURES)
    external = pd.read_csv(EXTERNAL_FEATURES)
    source = source[(source["label_binary"].isin(["positive", "negative"])) & (source["modality"] == "cough") & (source["split"] == "train")].copy()
    external = external[(external["label_binary"].isin(["positive", "negative"])) & (external["modality"] == "cough")].copy()
    if source.empty or external.empty:
        raise ValueError("Need non-empty source train cough rows and external labeled cough rows.")
    x_source, x_external, columns = harmonize_feature_columns(source, external)
    model = make_model("logistic_regression")
    model.fit(x_source, labels_to_binary(source["label_binary"]))
    probabilities = model.predict_proba(x_external)[:, 1]
    cross_predictions = pd.DataFrame({
        "recording_id": external["recording_id"].to_numpy(),
        "participant_id": external["participant_id"].to_numpy(),
        "dataset": external.get("dataset", pd.Series(["external"] * len(external))).to_numpy(),
        "modality": "cough",
        "label_binary": external["label_binary"].to_numpy(),
        "split": "external",
        "model_name": "logistic_regression",
        "probability": probabilities,
    })
    cross_metrics = evaluate_predictions(cross_predictions)
    cross_metrics["source_rows"] = len(source)
    cross_metrics["external_rows"] = len(external)
    cross_metrics["n_features"] = len(columns)
    save_table(cross_predictions, "data/outputs/metrics/cross_dataset_predictions.csv")
    save_table(cross_metrics, "data/outputs/metrics/cross_dataset_metrics.csv")
    display(cross_metrics)
else:
    print("RUN_CROSS_DATASET is False; skipping external evaluation.")

## Paper Metric Table

This consolidates available metric CSVs into a compact table for the report or manuscript. Missing files are skipped; run it after model/fusion/cross-dataset metrics exist.

In [ ]:
from covid_audio_btp.reporting import build_paper_metric_table, read_existing_csvs

metric_paths = [
    METRICS_DIR / "ml_baseline_metrics.csv",
    METRICS_DIR / "cnn_metrics.csv",
    METRICS_DIR / "calibration_metrics.csv",
    METRICS_DIR / "fusion_metrics.csv",
    METRICS_DIR / "metadata_baseline_metrics.csv",
    METRICS_DIR / "cross_dataset_metrics.csv",
]
ci_paths = [
    METRICS_DIR / "quality_weighted_fusion_bootstrap_ci.csv",
    METRICS_DIR / "cross_dataset_bootstrap_ci.csv",
]
combined_metrics = read_existing_csvs(metric_paths)
combined_ci = read_existing_csvs(ci_paths)
if combined_metrics.empty:
    print("No metric tables found yet; skipping paper metric table.")
else:
    group_columns = [col for col in ["table_source", "model_name", "model", "modality", "fusion_method", "dataset", "split"] if col in combined_metrics.columns]
    paper_metrics = build_paper_metric_table(combined_metrics, ci_table=combined_ci, group_columns=group_columns)
    save_table(combined_metrics, "reports/tables/paper_metric_table_raw.csv")
    save_table(paper_metrics, "reports/tables/paper_metric_table.csv")
    display(paper_metrics)

## Paper Asset Checklist

Before writing results, confirm these files exist: metadata baseline metrics, quality-weighted fusion predictions, abstention coverage curve, bootstrap confidence intervals, and cross-dataset metrics if enabled.

In [ ]:
final_status = check_artifacts([
    "data/outputs/metrics/metadata_baseline_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_predictions.csv",
    "data/outputs/metrics/abstention_coverage_curve.csv",
    "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv",
    "data/outputs/metrics/cross_dataset_metrics.csv",
])
display(final_status)